In [2]:
import torch
import torch.nn as nn
import numpy as np
import os

# ===== KROK 1: MODEL CLASS =====
print("=" * 80)
print("KONWERSJA PyTorch → ONNX")
print("=" * 80)

class MoleculeLogPPredictor(nn.Module):
    def __init__(self, input_size=2054, dropout_rate=0.3):
        super(MoleculeLogPPredictor, self).__init__()
        
        self.fc1 = nn.Linear(input_size, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout3 = nn.Dropout(dropout_rate)
        
        self.fc4 = nn.Linear(128, 1)
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout3(x)
        
        x = self.fc4(x)
        
        return x

print("\n✅ Model class zdefiniowany")

# ===== KROK 2: ZAŁADUJ MODEL PYTORCH =====
print(f"\n{'-' * 80}")
print(f"ZAŁADUJ WYTRENOWANY MODEL")
print(f"{'-' * 80}")

device = torch.device('cpu')
model = MoleculeLogPPredictor(input_size=2054)

model_path = r"C:\Users\slast\PYTHON\0_projekty do portfolio\07_pytorch_cl\checkpoints\best_model.pt"
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()

print(f"\n✅ Model załadowany z: {model_path}")
print(f"Device: {device}")

# ===== KROK 3: UTWÓRZ DUMMY INPUT =====
print(f"\n{'-' * 80}")
print(f"TESTOWANIE MODELU")
print(f"{'-' * 80}")

# Dummy input (2054 features: 2048 Morgan FP + 6 numeric)
dummy_input = torch.randn(1, 2054, dtype=torch.float32)

print(f"\nDummy input shape: {dummy_input.shape}")
print(f"Dummy input dtype: {dummy_input.dtype}")

# Test prediction
with torch.no_grad():
    test_output = model(dummy_input)
    print(f"Model output shape: {test_output.shape}")
    print(f"Model output value: {test_output.item():.4f}")

print(f"\n✅ Model działa prawidłowo")

# ===== KROK 4: KONWERSJA DO ONNX =====
print(f"\n{'-' * 80}")
print(f"KONWERSJA DO ONNX")
print(f"{'-' * 80}")

onnx_path = r"C:\Users\slast\PYTHON\0_projekty do portfolio\07_pytorch_cl\model.onnx"

# Parametry ONNX
input_names = ['features']
output_names = ['logp']

print(f"\nParametry konwersji:")
print(f"  Input names: {input_names}")
print(f"  Output names: {output_names}")
print(f"  Opset version: 12")

try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes={
            'features': {0: 'batch_size'},
            'logp': {0: 'batch_size'}
        },
        opset_version=12,
        do_constant_folding=True,
        verbose=False
    )
    
    print(f"\n✅ Konwersja POWIODŁA SIĘ!")
    print(f"Plik ONNX zapisany: {onnx_path}")
    
    # Sprawdź rozmiar pliku
    file_size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"Rozmiar pliku: {file_size:.2f} MB")
    
except Exception as e:
    print(f"\n❌ Błąd konwersji: {str(e)}")
    print(f"Spróbuj zainstalować: pip install onnx")

# ===== KROK 5: WERYFIKACJA ONNX =====
print(f"\n{'-' * 80}")
print(f"WERYFIKACJA ONNX MODELU")
print(f"{'-' * 80}")

try:
    import onnx
    import onnxruntime as ort
    
    # Załaduj ONNX model
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f"\n✅ ONNX model jest poprawny")
    
    # Załaduj inference session
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    print(f"✅ ONNX Runtime session utworzona")
    
    # Test inference
    dummy_numpy = dummy_input.numpy()
    outputs = sess.run(None, {'features': dummy_numpy})
    
    print(f"\n✅ ONNX Inference test:")
    print(f"  Output shape: {outputs[0].shape}")
    print(f"  Output value: {outputs[0][0][0]:.4f}")
    
    # Porównaj z PyTorch
    pytorch_output = model(dummy_input).detach().numpy()
    onnx_output = outputs[0]
    
    diff = np.abs(pytorch_output - onnx_output).max()
    print(f"\n✅ Porównanie PyTorch vs ONNX:")
    print(f"  Max difference: {diff:.8f}")
    
    if diff < 1e-5:
        print(f"  ✅ Wyniki są identyczne!")
    else:
        print(f"  ⚠️  Niewielka różnica (normalne)")
    
except ImportError:
    print(f"\n⚠️  ONNX Runtime nie zainstalowany")
    print(f"Zainstaluj: pip install onnx onnxruntime")
except Exception as e:
    print(f"\n❌ Błąd weryfikacji: {str(e)}")

# ===== KROK 6: PODSUMOWANIE =====
print(f"\n{'-' * 80}")
print(f"PODSUMOWANIE")
print(f"{'-' * 80}")

print(f"""
✅ KONWERSJA ZAKOŃCZONA!

📊 Model ONNX:
   Input: 2054 features (Morgan FP 2048 + numeric 6)
   Output: 1 (logP)
   Format: ONNX (universal, nie wymaga PyTorch!)
   
📁 Plik:
   {onnx_path}
   Rozmiar: ~{file_size:.1f} MB
   
🚀 Następne kroki:
   1. Zainstaluj ONNX Runtime
   2. Użyj model.onnx w aplikacji Streamlit
   3. Deploy na Streamlit Community Cloud
   
✨ Korzyści ONNX:
   ✅ Nie wymaga PyTorch
   ✅ Szybszy inference
   ✅ Mniejszy rozmiar
   ✅ Uniwersalny format (works everywhere!)
""")

print(f"\n✅ ZMIENNE DOSTĘPNE:")
print(f"   - model (PyTorch)")
print(f"   - onnx_path: {onnx_path}")
print(f"   - sess (ONNX Runtime session)")

KONWERSJA PyTorch → ONNX

✅ Model class zdefiniowany

--------------------------------------------------------------------------------
ZAŁADUJ WYTRENOWANY MODEL
--------------------------------------------------------------------------------

✅ Model załadowany z: C:\Users\slast\PYTHON\0_projekty do portfolio\07_pytorch_cl\checkpoints\best_model.pt
Device: cpu

--------------------------------------------------------------------------------
TESTOWANIE MODELU
--------------------------------------------------------------------------------

Dummy input shape: torch.Size([1, 2054])
Dummy input dtype: torch.float32
Model output shape: torch.Size([1, 1])
Model output value: 8.7845

✅ Model działa prawidłowo

--------------------------------------------------------------------------------
KONWERSJA DO ONNX
--------------------------------------------------------------------------------

Parametry konwersji:
  Input names: ['features']
  Output names: ['logp']
  Opset version: 12

✅ Konwersja